In [2]:
import requests
import pandas as pd
from pathlib import Path
import json

URL = "https://ws.audioscrobbler.com/2.0/?"
page=1
params = {
    "method": "user.getrecenttracks",
    "user": "joaoantonio0402",
    "limit": 200,
    "page": page,
    "extended": 0,
    "api_key": "71d883bfc3d9583a390b78c46f19f2e4",
    "format": "json"
}
response = requests.get(URL, params=params)
data = response.json()

In [4]:
total_pages = int(data["recenttracks"]["@attr"]["totalPages"])

print("Total pages:", total_pages)

# Diretório para os dados raw
Path("raw").mkdir(exist_ok=True)

# Salvar cada página exatamente como veio da API
for page in range(1, total_pages + 1):

    print(f"Buscando página {page}/{total_pages}")

    params["page"] = page

    response = requests.get(URL, params=params)
    response.raise_for_status()

    data = response.json()

    with open(f"raw/lastfm_page_{page}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

Total pages: 128
Buscando página 1/128
Buscando página 2/128
Buscando página 3/128
Buscando página 4/128
Buscando página 5/128
Buscando página 6/128
Buscando página 7/128
Buscando página 8/128
Buscando página 9/128
Buscando página 10/128
Buscando página 11/128
Buscando página 12/128
Buscando página 13/128
Buscando página 14/128
Buscando página 15/128
Buscando página 16/128
Buscando página 17/128
Buscando página 18/128
Buscando página 19/128
Buscando página 20/128
Buscando página 21/128
Buscando página 22/128
Buscando página 23/128
Buscando página 24/128
Buscando página 25/128
Buscando página 26/128
Buscando página 27/128
Buscando página 28/128
Buscando página 29/128
Buscando página 30/128
Buscando página 31/128
Buscando página 32/128
Buscando página 33/128
Buscando página 34/128
Buscando página 35/128
Buscando página 36/128
Buscando página 37/128
Buscando página 38/128
Buscando página 39/128
Buscando página 40/128
Buscando página 41/128
Buscando página 42/128
Buscando página 43/128
Bus

In [3]:
raw_dir = Path("raw")

json_files = sorted(raw_dir.glob("*.json"))

print(f"Arquivos encontrados: {len(json_files)}")

all_tracks = []

for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    tracks = data["recenttracks"]["track"]

    all_tracks.extend(tracks)

print(f"Total de registros: {len(all_tracks)}")

Arquivos encontrados: 128
Total de registros: 25415


In [ ]:
with open(f"data/processed/spotify/lastfm_recenttracks.json", "w", encoding="utf-8") as f:
    json.dump(all_tracks, f, ensure_ascii=False, indent=2)

In [5]:
from datetime import datetime, timezone

def parse_track(track):

    artist = track.get("artist", {})
    album = track.get("album", {})
    date = track.get("date", {})

    return {
        "track_mbid": track.get("mbid") or None,
        "track_name": track.get("name"),

        "artist_mbid": artist.get("mbid") or None,
        "artist_name": artist.get("#text"),

        "album_mbid": album.get("mbid") or None,
        "album_name": album.get("#text"),

        "timestamp_uts": int(date["uts"]) if date.get("uts") else None,
        "timestamp_utc": (
            datetime.fromtimestamp(int(date["uts"]), tz=timezone.utc)
            if date.get("uts") is not None
            else None
        ),

        "track_url": track.get("url"),
        "streamable": track.get("streamable"),
    }

In [6]:
raw_dir = Path("raw")

rows = []

for file in sorted(raw_dir.glob("*.json")):

    #print(f"Processando {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    tracks = data["recenttracks"]["track"]

    for track in tracks:
        rows.append(parse_track(track))


df = pd.DataFrame(rows)

In [7]:
df.head()

,track_mbid,track_name,artist_mbid,artist_name,album_mbid,album_name,timestamp_uts,timestamp_utc,track_url,streamable
0,03c49090-02c1-4481-b4ef-e737190fbeb6,Professional,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,16ff2d98-395c-4c87-aac8-cc4d62a853ec,Kiss Land,1786559671,2026-08-12 18:34:31+00:00,https://www.last.fm/music/The+Weeknd/_/Profess...,0
1,None,Vampiro,None,Matuê,None,Vampiro,1786504142,2026-08-12 03:09:02+00:00,https://www.last.fm/music/Matu%C3%AA/_/Vampiro,0
2,0d305f38-4094-4c98-a6ef-455e126edb6d,Heartbeat,7fb57fba-a6ef-44c2-abab-2fa3bdee607e,Childish Gambino,0e407fbd-58c1-45b3-adc5-a6ac5d945509,Camp,1786496834,2026-08-12 01:07:14+00:00,https://www.last.fm/music/Childish+Gambino/_/H...,0
3,None,When I’m Home,8dc08b1f-e393-4f85-a5dd-300f7693a8b8,James Blake,None,The Odyssey (Original Motion Picture Soundtrack),1786496562,2026-08-12 01:02:42+00:00,https://www.last.fm/music/James+Blake/_/When+I...,0
4,None,Both (feat. Drake),36494952-f434-45d8-a958-8b4acdbcf8a8,Gucci Mane,1bb059bb-47a5-4adc-a16e-ef1b30bc9610,The Return of East Atlanta Santa,1786488104,2026-08-11 22:41:44+00:00,https://www.last.fm/music/Gucci+Mane/_/Both+(f...,0


In [8]:
df.shape

(25415, 10)

In [11]:
df.value_counts("artist_name")

artist_name
The Weeknd           3706
Drake                1479
Lana Del Rey         1258
The Neighbourhood     927
Radiohead             852
                     ... 
GORDÃO DO PC            1
Galantis                1
trxndsetter             1
undeadprincess          1
Yung Bleu               1
Name: count, Length: 885, dtype: int64

In [13]:
dim_artist_df = (
    df[
        ["artist_mbid", "artist_name"]
    ]
    .drop_duplicates(subset="artist_name")
    .copy()
)

dim_artist_df.head()

,artist_mbid,artist_name
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd
1,None,Matuê
2,7fb57fba-a6ef-44c2-abab-2fa3bdee607e,Childish Gambino
3,8dc08b1f-e393-4f85-a5dd-300f7693a8b8,James Blake
4,36494952-f434-45d8-a958-8b4acdbcf8a8,Gucci Mane


In [19]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

DATABASE_URL = (
    "postgresql://postgres:1234"
    "@localhost:5432/google_and_spotify"
)


engine = create_engine(
    DATABASE_URL
)

with engine.connect() as conn:
    print("Conectado!")

    SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

Conectado!


In [25]:
dim_artist_df.to_sql("dim_artist", con=engine, if_exists="append", index=False)

885

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from schema import DimArtist

session = SessionLocal()

artist_map = {
    artist.artist_name: artist.artist_id
    for artist in session.query(DimArtist).all()
}

df["artist_id"] = df["artist_name"].map(artist_map)


In [34]:
dim_album_df = (
    df[
        ["album_mbid", "album_name", "artist_id"]
    ]
    .drop_duplicates(subset="album_name")
    .copy()
)

dim_album_df.head()

,album_mbid,album_name,artist_id
0,16ff2d98-395c-4c87-aac8-cc4d62a853ec,Kiss Land,1
1,None,Vampiro,2
2,0e407fbd-58c1-45b3-adc5-a6ac5d945509,Camp,3
3,None,The Odyssey (Original Motion Picture Soundtrack),4
4,1bb059bb-47a5-4adc-a16e-ef1b30bc9610,The Return of East Atlanta Santa,5


In [35]:
dim_album_df.to_sql("dim_album", con=engine, if_exists="append", index=False)

981

In [36]:
from schema import DimAlbum

album_map = {
    album.album_name: album.album_id
    for album in session.query(DimAlbum).all()
}

df["album_id"] = df["album_name"].map(album_map)

In [38]:
dim_track_df = (
    df[
        ["track_mbid", "track_name", "artist_id", "album_id"]
    ]
    .drop_duplicates(subset="track_name")
    .copy()
)

dim_track_df.head()

,track_mbid,track_name,artist_id,album_id
0,03c49090-02c1-4481-b4ef-e737190fbeb6,Professional,1,2
1,None,Vampiro,2,3
2,0d305f38-4094-4c98-a6ef-455e126edb6d,Heartbeat,3,4
3,None,When I’m Home,4,5
4,None,Both (feat. Drake),5,6


In [39]:
dim_track_df.to_sql("dim_track", con=engine, if_exists="append", index=False)

449

In [40]:
from schema import DimTrack

track_map = {
    track.track_name: track.track_id
    for track in session.query(DimTrack).all()
}

df["track_id"] = df["track_name"].map(track_map)

In [41]:
df.head()

,track_mbid,track_name,artist_mbid,artist_name,album_mbid,album_name,timestamp_uts,timestamp_utc,track_url,streamable,artist_id,album_id,track_id
0,03c49090-02c1-4481-b4ef-e737190fbeb6,Professional,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,16ff2d98-395c-4c87-aac8-cc4d62a853ec,Kiss Land,1786559671,2026-08-12 18:34:31+00:00,https://www.last.fm/music/The+Weeknd/_/Profess...,0,1,2,1
1,None,Vampiro,None,Matuê,None,Vampiro,1786504142,2026-08-12 03:09:02+00:00,https://www.last.fm/music/Matu%C3%AA/_/Vampiro,0,2,3,2
2,0d305f38-4094-4c98-a6ef-455e126edb6d,Heartbeat,7fb57fba-a6ef-44c2-abab-2fa3bdee607e,Childish Gambino,0e407fbd-58c1-45b3-adc5-a6ac5d945509,Camp,1786496834,2026-08-12 01:07:14+00:00,https://www.last.fm/music/Childish+Gambino/_/H...,0,3,4,3
3,None,When I’m Home,8dc08b1f-e393-4f85-a5dd-300f7693a8b8,James Blake,None,The Odyssey (Original Motion Picture Soundtrack),1786496562,2026-08-12 01:02:42+00:00,https://www.last.fm/music/James+Blake/_/When+I...,0,4,5,4
4,None,Both (feat. Drake),36494952-f434-45d8-a958-8b4acdbcf8a8,Gucci Mane,1bb059bb-47a5-4adc-a16e-ef1b30bc9610,The Return of East Atlanta Santa,1786488104,2026-08-11 22:41:44+00:00,https://www.last.fm/music/Gucci+Mane/_/Both+(f...,0,5,6,5


In [42]:
fact_listening_df = (
    df[
        ["track_id", "timestamp_utc", "timestamp_uts"]
    ]
    .copy()
)

fact_listening_df.head()

,track_id,timestamp_utc,timestamp_uts
0,1,2026-08-12 18:34:31+00:00,1786559671
1,2,2026-08-12 03:09:02+00:00,1786504142
2,3,2026-08-12 01:07:14+00:00,1786496834
3,4,2026-08-12 01:02:42+00:00,1786496562
4,5,2026-08-11 22:41:44+00:00,1786488104


In [44]:
fact_listening_df.to_sql("fact_listening", con=engine, if_exists="append", index=False)

415

In [ ]:
df.to_csv("data/processed/spotify/lastfm_recenttracks.csv", index=False, encoding="utf-8")

In [45]:
import requests

In [ ]:
def search_recco(track_name):
    url = "https://api.reccobeats.com/v1/track/search"
    
    params = {
        "searchText": track_name
    }
    
    response = requests.get(url, params=params)
    
    return response.json()

df_recco = df.copy().drop_duplicates(subset=["track_name", "artist_name"])

df_recco = df_recco.head(100)

df_recco["recco_result"] = df_recco["track_name"].apply(search_recco)

df_recco.head()

,track_mbid,track_name,artist_mbid,artist_name,album_mbid,album_name,timestamp_uts,timestamp_utc,track_url,streamable,artist_id,album_id,track_id,recco_result
0,03c49090-02c1-4481-b4ef-e737190fbeb6,Professional,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,16ff2d98-395c-4c87-aac8-cc4d62a853ec,Kiss Land,1786559671,2026-08-12 18:34:31+00:00,https://www.last.fm/music/The+Weeknd/_/Profess...,0,1,2,1,{'content': [{'id': 'c9ea3a2d-dc6a-4462-9aa6-5...
1,None,Vampiro,None,Matuê,None,Vampiro,1786504142,2026-08-12 03:09:02+00:00,https://www.last.fm/music/Matu%C3%AA/_/Vampiro,0,2,3,2,{'content': [{'id': '50b3c5ea-9745-4f64-9f54-4...
2,0d305f38-4094-4c98-a6ef-455e126edb6d,Heartbeat,7fb57fba-a6ef-44c2-abab-2fa3bdee607e,Childish Gambino,0e407fbd-58c1-45b3-adc5-a6ac5d945509,Camp,1786496834,2026-08-12 01:07:14+00:00,https://www.last.fm/music/Childish+Gambino/_/H...,0,3,4,3,{'content': [{'id': 'f7be9aff-6c52-4667-be35-6...
3,None,When I’m Home,8dc08b1f-e393-4f85-a5dd-300f7693a8b8,James Blake,None,The Odyssey (Original Motion Picture Soundtrack),1786496562,2026-08-12 01:02:42+00:00,https://www.last.fm/music/James+Blake/_/When+I...,0,4,5,4,"{'content': [], 'page': 0, 'size': 25, 'totalE..."
4,None,Both (feat. Drake),36494952-f434-45d8-a958-8b4acdbcf8a8,Gucci Mane,1bb059bb-47a5-4adc-a16e-ef1b30bc9610,The Return of East Atlanta Santa,1786488104,2026-08-11 22:41:44+00:00,https://www.last.fm/music/Gucci+Mane/_/Both+(f...,0,5,6,5,{'content': [{'id': '3c2c6fb7-b655-4ee1-8306-a...


In [118]:
# df_recco = df_recco.iloc[1:3]
# df_recco

In [119]:
df_recco["recco_result"] = df_recco["recco_result"].apply(
    lambda x: x["content"] if x else []
)

df_recco = df_recco.explode(
    "recco_result",
    ignore_index=True
)

recco_data = pd.json_normalize(
    df_recco["recco_result"]
)

df_recco = pd.concat(
    [
        df_recco.drop(columns="recco_result").reset_index(drop=True),
        recco_data["artists"].reset_index(drop=True)
    ],
    axis=1
)

df_recco.head()

KeyError: 'content'

In [ ]:
df_recco["artist_name_recco"] = df_recco["artists"].apply(
    lambda x: ", ".join(a["name"] for a in x) if isinstance(x, list) else None
)

In [ ]:
df_recco["certo"] = df_recco["artist_name"].where(
    df_recco["artist_name"] == df_recco["artist_name_recco"]
)

In [ ]:
df_recco = df_recco.dropna(subset=["certo"])
df_recco = df_recco.drop_duplicates(subset=["track_name", "artist_name", "artist_name_recco"])
df_recco

,track_mbid,track_name,artist_mbid,artist_name,album_mbid,album_name,timestamp_uts,timestamp_utc,track_url,streamable,artist_id,album_id,track_id,artists,artist_name_recco,certo
0,03c49090-02c1-4481-b4ef-e737190fbeb6,Professional,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,16ff2d98-395c-4c87-aac8-cc4d62a853ec,Kiss Land,1786559671,2026-08-12 18:34:31+00:00,https://www.last.fm/music/The+Weeknd/_/Profess...,0,1,2,1,[{'id': '9451b6b2-8746-4d43-abd7-c355ed1e3048'...,The Weeknd,The Weeknd
134,6cf3c4d4-c2c2-3d38-adb1-a66c70636864,Novacane,e520459c-dff4-491d-a6e4-c97be35e0044,Frank Ocean,16f3fcf2-4511-4c8c-93bb-b4c8910aa9db,Novacane,1786486021,2026-08-11 22:07:01+00:00,https://www.last.fm/music/Frank+Ocean/_/Novacane,0,8,9,8,[{'id': '26b85419-be43-4509-bf41-4e7f7a4a89dc'...,Frank Ocean,Frank Ocean
260,None,Meet Me Halfway,d5be5333-4171-427e-8e12-732087c6b78e,Black Eyed Peas,None,The E.N.D. (The Energy Never Dies) [Deluxe Ver...,1786483796,2026-08-11 21:29:56+00:00,https://www.last.fm/music/Black+Eyed+Peas/_/Me...,0,11,15,13,[{'id': '092ac14e-36df-4051-b923-faa352d99c0e'...,Black Eyed Peas,Black Eyed Peas
285,703d59cc-8008-4220-99da-43f32185a944,Pretty When You Cry,b7539c32-53e7-4908-bda3-81449c367da6,Lana Del Rey,None,Ultraviolence (Deluxe),1786483407,2026-08-11 21:23:27+00:00,https://www.last.fm/music/Lana+Del+Rey/_/Prett...,0,12,16,14,[{'id': 'fb5d70f5-57b8-44ff-a700-71badf22839b'...,Lana Del Rey,Lana Del Rey
360,3f111917-ccbd-43fd-bca1-bdd6379ad175,THANK GOD,e4a51f17-a57b-47b1-b37b-f552d0f8e9e6,Travis Scott,d6091fdc-58a1-4659-8d31-7b2be5021e20,UTOPIA,1786482083,2026-08-11 21:01:23+00:00,https://www.last.fm/music/Travis+Scott/_/THANK...,0,14,19,17,[{'id': 'aa6e2a7e-28d9-4576-b2ef-176243a579a2'...,Travis Scott,Travis Scott
386,4bcb550c-9160-45ed-9cbe-d6771ad96c9d,CHIHIRO,f4abc0b5-3f7a-4eff-8f78-ac078dbce533,Billie Eilish,43869ac5-c515-41a4-8ce1-d4a0c67e30c1,HIT ME HARD AND SOFT,1786481696,2026-08-11 20:54:56+00:00,https://www.last.fm/music/Billie+Eilish/_/CHIHIRO,0,16,21,19,[{'id': 'bf2c5ca2-f233-4846-b804-3a0b4a6a9909'...,Billie Eilish,Billie Eilish
436,None,Cheyenne,None,Trilucid,None,Cheyenne,1786481225,2026-08-11 20:47:05+00:00,https://www.last.fm/music/Trilucid/_/Cheyenne,0,18,23,21,[{'id': '1ee4b6fa-b6d9-4d62-8ca0-578bd1f61e61'...,Trilucid,Trilucid
462,34854dca-e909-4271-8bb1-00f2cf5a4d53,Xerces,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,45127ff8-0a5e-41a8-94e4-53b0184b0fe8,Saturday Night Wrist,1786480726,2026-08-11 20:38:46+00:00,https://www.last.fm/music/Deftones/_/Xerces,0,20,25,23,[{'id': '7d9d2a48-fb31-4e16-a7fb-1ca42830d2d2'...,Deftones,Deftones
